# Tableaux récapitulatifs (1)

Mise en forme des tableaux de résultats du mémoire.

> **Exécution** : notebook conçu pour Google Colab. Les chemins pointent vers Google Drive (`/content/drive/MyDrive/bug/...`). Voir [`data/README.md`](../../data/README.md) pour obtenir les données et adapter les chemins.

*Nom d'origine dans Colab : `Untitled92`*


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
"""
================================================================================
ANNEXE A.3 - PATTERNS D'ÉVOLUTION TEMPORELLE DES MÉTRIQUES
================================================================================

Code Python pour Google Colab
Génère les visualisations des 4 patterns d'évolution temporelle identifiés
dans le corpus PROMISE (11 projets Java)

Auteure: Mireille Kafando
Mémoire: Prédiction de défauts logiciels avec apprentissage profond
Institution: UQTR

Patterns identifiés:
1. Stables: Ant, Log4j, POI, Ivy
2. Croissance: Lucene, Xalan, Synapse
3. Oscillatoires: Velocity, Camel
4. Décroissance: JEdit, Xerces
================================================================================
"""

# ============================================================================
# INSTALLATION ET IMPORTATION DES BIBLIOTHÈQUES
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

print("✅ Bibliothèques importées avec succès")

# ============================================================================
# CHARGEMENT DES DONNÉES
# ============================================================================

# Données récapitulatives par version (à adapter selon votre environnement)
# Option 1: Depuis Google Drive (recommandé pour Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# data = pd.read_csv('/content/drive/MyDrive/tableau_recapitulatif_1.csv')

# Option 2: Upload manuel dans Colab
# from google.colab import files
# uploaded = files.upload()
# data = pd.read_csv('tableau_recapitulatif_1.csv')

# Option 3: Données simulées (pour test)
# Pour l'exemple, je vais créer les données basées sur le tableau fourni
data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'Version': ['ant-1.3', 'ant-1.4', 'ant-1.5', 'ant-1.6', 'ant-1.7',
                'camel-1.0', 'camel-1.2', 'camel-1.4', 'camel-1.6',
                'ivy-1.0', 'ivy-1.1', 'ivy-1.2',
                'jedit-3.2', 'jedit-4.0', 'jedit-4.1', 'jedit-4.2', 'jedit-4.3',
                'log4j-1.0', 'log4j-1.1', 'log4j-1.2',
                'lucene-2.0', 'lucene-2.2', 'lucene-2.4',
                'poi-1.5', 'poi-2.0', 'poi-2.5', 'poi-3.0',
                'synapse-1.0', 'synapse-1.1', 'synapse-1.2',
                'velocity-1.4', 'velocity-1.5', 'velocity-1.6',
                'xalan-2.4', 'xalan-2.5', 'xalan-2.6', 'xalan-2.7',
                'xerces-1.1', 'xerces-1.2', 'xerces-1.3', 'xerces-1.4.4'],
    'Classes': [125, 178, 293, 351, 745,
                339, 608, 872, 965,
                111, 241, 352,
                272, 306, 312, 367, 492,
                135, 109, 205,
                195, 247, 340,
                237, 314, 385, 442,
                157, 222, 256,
                196, 214, 229,
                723, 803, 885, 909,
                162, 440, 453, 588],
    'Défauts': [33, 47, 35, 184, 338,
                14, 522, 335, 500,
                233, 18, 56,
                382, 226, 217, 106, 12,
                61, 86, 498,
                268, 414, 632,
                342, 39, 496, 500,
                21, 99, 145,
                210, 331, 190,
                156, 531, 625, 1213,
                167, 115, 193, 1596],
    'Taux Défauts (%)': [26.4, 26.4, 11.9, 52.4, 45.4,
                         4.1, 85.9, 38.4, 51.8,
                         209.9, 7.5, 15.9,
                         140.4, 73.9, 69.6, 28.9, 2.4,
                         45.2, 78.9, 242.9,
                         137.4, 167.6, 185.9,
                         144.3, 12.4, 128.8, 113.1,
                         13.4, 44.6, 56.6,
                         107.1, 154.7, 83.0,
                         21.6, 66.1, 70.6, 133.4,
                         103.1, 26.1, 42.6, 271.4],
    'LOC Moyen': [302, 304, 297, 323, 280,
                  99, 109, 112, 117,
                  246, 246, 249,
                  474, 473, 491, 465, 411,
                  160, 183, 186,
                  259, 257, 303,
                  234, 297, 311, 293,
                  183, 191, 209,
                  264, 248, 249,
                  311, 380, 465, 471,
                  560, 362, 369, 240],
    'WMC Moyen': [10.6, 10.5, 10.1, 11.1, 11.1,
                  8.1, 8.3, 8.5, 8.6,
                  10.6, 10.7, 11.3,
                  12.5, 12.9, 13.1, 13.2, 12.4,
                  6.6, 7.8, 8.4,
                  9.3, 9.4, 10.4,
                  13.4, 14.3, 14.3, 13.5,
                  8.1, 7.3, 7.6,
                  8.8, 9.0, 9.0,
                  11.4, 11.3, 11.0, 10.9,
                  11.4, 11.3, 11.4, 9.9],
    'CBO Moyen': [10.4, 10.8, 10.6, 11.5, 11.0,
                  10.0, 10.1, 10.8, 11.1,
                  10.4, 11.3, 13.2,
                  12.0, 12.4, 13.0, 14.1, 14.3,
                  6.9, 6.9, 7.6,
                  9.8, 10.0, 10.8,
                  8.3, 8.7, 9.0, 10.1,
                  13.2, 12.5, 12.8,
                  10.6, 10.6, 10.8,
                  14.5, 12.9, 12.1, 12.0,
                  6.1, 4.9, 5.1, 6.3],
    'RFC Moyen': [34.4, 33.8, 30.7, 34.2, 34.4,
                  19.6, 20.2, 21.2, 21.4,
                  32.4, 31.3, 34.0,
                  37.7, 38.2, 39.9, 41.0, 39.8,
                  21.4, 23.2, 25.2,
                  23.1, 22.9, 25.2,
                  27.6, 29.6, 30.9, 30.4,
                  29.8, 28.6, 29.5,
                  23.6, 22.6, 23.0,
                  30.2, 29.6, 29.3, 29.2,
                  23.3, 21.2, 21.7, 19.2]
}

data = pd.DataFrame(data_dict)

# Ajouter un index de version numérique pour chaque projet
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

print("\n✅ Données chargées avec succès")
print(f"\nNombre de projets: {data['Projet'].nunique()}")
print(f"Nombre total de versions: {len(data)}")
print(f"\nProjets analysés: {', '.join(sorted(data['Projet'].unique()))}")

# ============================================================================
# DÉFINITION DES PATTERNS
# ============================================================================

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

# Couleurs par pattern
pattern_colors = {
    'Stables': '#2E86AB',        # Bleu
    'Croissance': '#06A77D',     # Vert
    'Oscillatoires': '#F18F01',  # Orange
    'Décroissance': '#C73E1D'    # Rouge
}

print("\n" + "="*80)
print("PATTERNS D'ÉVOLUTION IDENTIFIÉS")
print("="*80)
for pattern_name, projects in patterns.items():
    print(f"\n{pattern_name}: {', '.join(projects)}")

# ============================================================================
# FONCTION DE CALCUL DES TENDANCES
# ============================================================================

def calculate_trend(projet_data, metric):
    """
    Calcule la tendance (pente) d'une métrique pour un projet.
    Retourne la pente de la régression linéaire.
    """
    x = np.arange(len(projet_data))
    y = projet_data[metric].values

    if len(x) < 2:
        return 0

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    return slope

# Calculer les tendances pour chaque projet
trends = {}
for projet in data['Projet'].unique():
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
    trends[projet] = {
        'LOC': calculate_trend(projet_data, 'LOC Moyen'),
        'WMC': calculate_trend(projet_data, 'WMC Moyen'),
        'Classes': calculate_trend(projet_data, 'Classes'),
        'Défauts': calculate_trend(projet_data, 'Taux Défauts (%)')
    }

print("\n" + "="*80)
print("TENDANCES CALCULÉES (pente de régression linéaire)")
print("="*80)
for projet, trend_values in trends.items():
    print(f"\n{projet}:")
    for metric, slope in trend_values.items():
        direction = "↑" if slope > 0 else "↓" if slope < 0 else "→"
        print(f"  {metric}: {slope:+.2f} {direction}")

# ============================================================================
# VISUALISATION 1: ÉVOLUTION PAR PATTERN (4 SOUS-GRAPHIQUES)
# ============================================================================

def plot_pattern_evolution(pattern_name, projects, metric='LOC Moyen',
                           ylabel='LOC Moyen', color='#2E86AB'):
    """
    Crée un graphique d'évolution pour un pattern donné.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

        ax.plot(projet_data['Version_Num'], projet_data[metric],
                marker='o', linewidth=2.5, markersize=8,
                label=projet, alpha=0.8)

    ax.set_xlabel('Version (ordre chronologique)', fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=13, fontweight='bold')
    ax.set_title(f'Pattern {pattern_name}: Évolution de {ylabel}',
                 fontsize=15, fontweight='bold', pad=20)
    ax.legend(loc='best', frameon=True, shadow=True)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    return fig

print("\n" + "="*80)
print("GÉNÉRATION DES VISUALISATIONS - ANNEXE A.3")
print("="*80)

# Créer les 4 graphiques pour LOC Moyen
figures_loc = {}
for pattern_name, projects in patterns.items():
    print(f"\n📊 Génération: Pattern {pattern_name} - LOC Moyen...")
    figures_loc[pattern_name] = plot_pattern_evolution(
        pattern_name, projects, 'LOC Moyen', 'LOC Moyen (lignes de code)',
        pattern_colors[pattern_name]
    )
    plt.show()

# ============================================================================
# VISUALISATION 2: ÉVOLUTION WMC (COMPLEXITÉ)
# ============================================================================

figures_wmc = {}
for pattern_name, projects in patterns.items():
    print(f"\n📊 Génération: Pattern {pattern_name} - WMC Moyen...")
    figures_wmc[pattern_name] = plot_pattern_evolution(
        pattern_name, projects, 'WMC Moyen', 'WMC Moyen (complexité)',
        pattern_colors[pattern_name]
    )
    plt.show()

# ============================================================================
# VISUALISATION 3: GRAPHIQUE COMBINÉ PAR PATTERN (2x2)
# ============================================================================

print("\n📊 Génération: Vue d'ensemble des 4 patterns...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.ravel()

for idx, (pattern_name, projects) in enumerate(patterns.items()):
    ax = axes[idx]

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2.5, markersize=8,
                label=projet, alpha=0.8)

    ax.set_xlabel('Version (ordre chronologique)', fontsize=12, fontweight='bold')
    ax.set_ylabel('LOC Moyen (lignes de code)', fontsize=12, fontweight='bold')
    ax.set_title(f'{pattern_name}',
                 fontsize=14, fontweight='bold', pad=15,
                 color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=True, fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('ANNEXE A.3 - Patterns d\'Évolution Temporelle des Métriques\n' +
             'Corpus PROMISE (11 Projets Java)',
             fontsize=16, fontweight='bold', y=0.998)
plt.tight_layout()
plt.show()

# ============================================================================
# VISUALISATION 4: HEATMAP DES VARIATIONS
# ============================================================================

print("\n📊 Génération: Heatmap des variations relatives...")

# Calculer les variations relatives pour chaque projet
variations = []
for projet in data['Projet'].unique():
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

    if len(projet_data) > 1:
        # Variation relative entre première et dernière version
        var_loc = ((projet_data.iloc[-1]['LOC Moyen'] - projet_data.iloc[0]['LOC Moyen'])
                   / projet_data.iloc[0]['LOC Moyen'] * 100)
        var_wmc = ((projet_data.iloc[-1]['WMC Moyen'] - projet_data.iloc[0]['WMC Moyen'])
                   / projet_data.iloc[0]['WMC Moyen'] * 100)
        var_classes = ((projet_data.iloc[-1]['Classes'] - projet_data.iloc[0]['Classes'])
                       / projet_data.iloc[0]['Classes'] * 100)

        variations.append({
            'Projet': projet,
            'LOC (%)': var_loc,
            'WMC (%)': var_wmc,
            'Classes (%)': var_classes
        })

df_variations = pd.DataFrame(variations)
df_variations = df_variations.set_index('Projet')

# Créer la heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_variations.T, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, cbar_kws={'label': 'Variation (%)'},
            linewidths=0.5, ax=ax)
ax.set_title('Variations Relatives des Métriques\n(Première vs Dernière Version)',
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Projets', fontsize=12, fontweight='bold')
ax.set_ylabel('Métriques', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# VISUALISATION 5: ÉVOLUTION NORMALISÉE (TOUS PROJETS)
# ============================================================================

print("\n📊 Génération: Évolution normalisée tous projets...")

# Normaliser les données (0-1) pour comparaison
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

metrics_to_plot = ['LOC Moyen', 'WMC Moyen', 'Classes', 'Taux Défauts (%)']
metric_labels = ['LOC Moyen Normalisé', 'WMC Moyen Normalisé',
                 'Nombre de Classes Normalisé', 'Taux de Défauts Normalisé']

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[idx // 2, idx % 2]

    for pattern_name, projects in patterns.items():
        for projet in projects:
            projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

            # Normalisation min-max
            values = projet_data[metric].values
            if values.max() > values.min():
                normalized = (values - values.min()) / (values.max() - values.min())
            else:
                normalized = values

            ax.plot(projet_data['Version_Num'], normalized,
                    marker='o', linewidth=2, markersize=6,
                    alpha=0.6, color=pattern_colors[pattern_name])

    # Ajouter les légendes des patterns
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, label=pattern)
                      for pattern, color in pattern_colors.items()]
    ax.legend(handles=legend_elements, loc='best', frameon=True, shadow=True)

    ax.set_xlabel('Version (ordre chronologique)', fontsize=11, fontweight='bold')
    ax.set_ylabel(label, fontsize=11, fontweight='bold')
    ax.set_title(label.replace(' Normalisé', ''), fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Évolution Normalisée des Métriques par Pattern\n' +
             '(Valeurs normalisées entre 0 et 1 pour comparaison)',
             fontsize=15, fontweight='bold', y=0.998)
plt.tight_layout()
plt.show()

# ============================================================================
# ANALYSE STATISTIQUE DES PATTERNS
# ============================================================================

print("\n" + "="*80)
print("ANALYSE STATISTIQUE DES PATTERNS")
print("="*80)

for pattern_name, projects in patterns.items():
    print(f"\n{'='*60}")
    print(f"Pattern: {pattern_name}")
    print(f"{'='*60}")

    pattern_data = data[data['Projet'].isin(projects)]

    print(f"\nNombre de projets: {len(projects)}")
    print(f"Nombre total de versions: {len(pattern_data)}")
    print(f"Versions par projet: {pattern_data.groupby('Projet').size().to_dict()}")

    print(f"\nStatistiques LOC Moyen:")
    print(f"  Moyenne: {pattern_data['LOC Moyen'].mean():.1f}")
    print(f"  Écart-type: {pattern_data['LOC Moyen'].std():.1f}")
    print(f"  Min: {pattern_data['LOC Moyen'].min():.1f}")
    print(f"  Max: {pattern_data['LOC Moyen'].max():.1f}")

    print(f"\nStatistiques WMC Moyen:")
    print(f"  Moyenne: {pattern_data['WMC Moyen'].mean():.1f}")
    print(f"  Écart-type: {pattern_data['WMC Moyen'].std():.1f}")

    # Calculer la variance entre versions pour chaque projet
    variances = []
    for projet in projects:
        projet_data = pattern_data[pattern_data['Projet'] == projet]
        if len(projet_data) > 1:
            var = projet_data['LOC Moyen'].var()
            variances.append(var)

    if variances:
        print(f"\nVariance intra-projet moyenne (LOC): {np.mean(variances):.1f}")
        print(f"  → Stabilité: {'Faible' if np.mean(variances) < 1000 else 'Moyenne' if np.mean(variances) < 5000 else 'Élevée'}")

# ============================================================================
# TABLEAU RÉCAPITULATIF
# ============================================================================

print("\n" + "="*80)
print("TABLEAU RÉCAPITULATIF - CARACTÉRISTIQUES DES PATTERNS")
print("="*80)

summary_data = []
for pattern_name, projects in patterns.items():
    pattern_data = data[data['Projet'].isin(projects)]

    # Tendances moyennes
    avg_trends = {
        'LOC': np.mean([trends[p]['LOC'] for p in projects]),
        'WMC': np.mean([trends[p]['WMC'] for p in projects]),
        'Classes': np.mean([trends[p]['Classes'] for p in projects])
    }

    summary_data.append({
        'Pattern': pattern_name,
        'Nb Projets': len(projects),
        'Nb Versions': len(pattern_data),
        'LOC Moyen': f"{pattern_data['LOC Moyen'].mean():.0f}",
        'Tendance LOC': f"{avg_trends['LOC']:+.1f}",
        'Tendance WMC': f"{avg_trends['WMC']:+.2f}",
        'Tendance Classes': f"{avg_trends['Classes']:+.0f}",
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

# ============================================================================
# EXPORT DES RÉSULTATS
# ============================================================================

print("\n" + "="*80)
print("EXPORT DES RÉSULTATS")
print("="*80)

# Sauvegarder le tableau récapitulatif
df_summary.to_csv('annexe_a3_patterns_summary.csv', index=False)
print("\n✅ Tableau récapitulatif sauvegardé: annexe_a3_patterns_summary.csv")

# Sauvegarder les tendances
df_trends = pd.DataFrame(trends).T
df_trends.to_csv('annexe_a3_tendances.csv')
print("✅ Tendances sauvegardées: annexe_a3_tendances.csv")

# Sauvegarder les variations
df_variations.to_csv('annexe_a3_variations.csv')
print("✅ Variations sauvegardées: annexe_a3_variations.csv")

print("\n" + "="*80)
print("GÉNÉRATION TERMINÉE - ANNEXE A.3")
print("="*80)
print("\n✅ Toutes les visualisations ont été générées avec succès!")
print("\n📊 Graphiques produits:")
print("   1. Évolution par pattern (LOC) - 4 graphiques")
print("   2. Évolution par pattern (WMC) - 4 graphiques")
print("   3. Vue d'ensemble 2x2 des patterns")
print("   4. Heatmap des variations relatives")
print("   5. Évolution normalisée (tous projets)")
print("\n📁 Fichiers CSV exportés:")
print("   - annexe_a3_patterns_summary.csv")
print("   - annexe_a3_tendances.csv")
print("   - annexe_a3_variations.csv")
print("\n" + "="*80)

# ============================================================================
# INTERPRÉTATION DES PATTERNS (POUR DOCUMENTATION)
# ============================================================================

print("\n" + "="*80)
print("INTERPRÉTATION DES PATTERNS OBSERVÉS")
print("="*80)

interpretations = {
    'Stables': """
    Les projets Ant, Log4j, POI et Ivy présentent des métriques relativement
    constantes entre versions successives. Cette stabilité suggère une
    architecture mature avec peu de refactoring majeur. La stratégie de
    standardisation globale (SG) devrait être efficace car les distributions
    de métriques restent cohérentes dans le temps.
    """,

    'Croissance': """
    Les projets Lucene, Xalan et Synapse montrent des tendances croissantes
    systématiques pour plusieurs métriques (LOC, nombre de classes). Cette
    croissance reflète l'ajout continu de nouvelles fonctionnalités. La
    stratégie par version (SV) est justifiée car elle s'adapte aux
    caractéristiques locales de chaque version en évolution.
    """,

    'Oscillatoires': """
    Les projets Velocity et Camel exhibent des variations périodiques avec
    des pics et des creux. Ces oscillations rendent problématiques les
    approches purement globales ou purement locales. La stratégie hybride
    SR-Robuste, combinant adaptation locale et cohérence globale, est
    particulièrement adaptée à ce pattern.
    """,

    'Décroissance': """
    Les projets JEdit et Xerces suggèrent des efforts d'amélioration continue
    de la qualité du code (refactoring, réduction de la complexité). La
    décroissance des métriques de complexité (LOC, WMC) indique une
    restructuration active. Ce pattern renforce l'importance du respect
    scrupuleux de la chronologie temporelle dans les protocoles de validation.
    """
}

for pattern_name, interpretation in interpretations.items():
    print(f"\n{'='*60}")
    print(f"{pattern_name}")
    print(f"{'='*60}")
    print(interpretation.strip())

print("\n" + "="*80)
print("FIN DE L'ANALYSE - ANNEXE A.3")
print("="*80)

In [ ]:
"""
================================================================================
ANNEXE A.3 - PATTERNS D'ÉVOLUTION TEMPORELLE - VERSION OPTIMISÉE WORD
================================================================================

Version optimisée pour intégration dans document Word
- Tailles adaptées au format A4 avec marges 2.5cm
- Résolution 300 DPI pour impression de qualité
- Export automatique PNG et PDF
- Polices lisibles en format réduit

Auteure: Mireille Kafando
Mémoire: Prédiction de défauts logiciels avec apprentissage profond
Institution: UQTR
================================================================================
"""

# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION OPTIMISÉE POUR WORD
# ============================================================================

# Configuration pour document Word (format A4, marges 2.5cm)
# Largeur maximale utilisable: ~6.5 pouces (~16.5 cm)

plt.style.use('seaborn-v0_8-whitegrid')  # Style plus propre pour impression

# Tailles optimisées pour Word
SINGLE_FIG_SIZE = (6.5, 4)      # Graphique individuel (largeur pleine page)
GRID_2x2_SIZE = (6.5, 8)        # Grille 2×2 (largeur pleine, hauteur double)
GRID_2x1_SIZE = (6.5, 6)        # Grille 2×1
DPI = 300                        # Résolution pour impression de qualité

# Configuration des polices (lisibles en format réduit)
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

print("✅ Configuration optimisée pour Word")
print(f"   - Largeur graphiques: 6.5 pouces (16.5 cm)")
print(f"   - Résolution: {DPI} DPI")
print(f"   - Format: PNG et PDF")

# ============================================================================
# CHARGEMENT DES DONNÉES
# ============================================================================

data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'Version': ['ant-1.3', 'ant-1.4', 'ant-1.5', 'ant-1.6', 'ant-1.7',
                'camel-1.0', 'camel-1.2', 'camel-1.4', 'camel-1.6',
                'ivy-1.0', 'ivy-1.1', 'ivy-1.2',
                'jedit-3.2', 'jedit-4.0', 'jedit-4.1', 'jedit-4.2', 'jedit-4.3',
                'log4j-1.0', 'log4j-1.1', 'log4j-1.2',
                'lucene-2.0', 'lucene-2.2', 'lucene-2.4',
                'poi-1.5', 'poi-2.0', 'poi-2.5', 'poi-3.0',
                'synapse-1.0', 'synapse-1.1', 'synapse-1.2',
                'velocity-1.4', 'velocity-1.5', 'velocity-1.6',
                'xalan-2.4', 'xalan-2.5', 'xalan-2.6', 'xalan-2.7',
                'xerces-1.1', 'xerces-1.2', 'xerces-1.3', 'xerces-1.4.4'],
    'LOC Moyen': [302, 304, 297, 323, 280, 99, 109, 112, 117, 246, 246, 249,
                  474, 473, 491, 465, 411, 160, 183, 186, 259, 257, 303,
                  234, 297, 311, 293, 183, 191, 209, 264, 248, 249,
                  311, 380, 465, 471, 560, 362, 369, 240],
    'WMC Moyen': [10.6, 10.5, 10.1, 11.1, 11.1, 8.1, 8.3, 8.5, 8.6,
                  10.6, 10.7, 11.3, 12.5, 12.9, 13.1, 13.2, 12.4,
                  6.6, 7.8, 8.4, 9.3, 9.4, 10.4, 13.4, 14.3, 14.3, 13.5,
                  8.1, 7.3, 7.6, 8.8, 9.0, 9.0, 11.4, 11.3, 11.0, 10.9,
                  11.4, 11.3, 11.4, 9.9],
    'CBO Moyen': [10.4, 10.8, 10.6, 11.5, 11.0, 10.0, 10.1, 10.8, 11.1,
                  10.4, 11.3, 13.2, 12.0, 12.4, 13.0, 14.1, 14.3,
                  6.9, 6.9, 7.6, 9.8, 10.0, 10.8, 8.3, 8.7, 9.0, 10.1,
                  13.2, 12.5, 12.8, 10.6, 10.6, 10.8, 14.5, 12.9, 12.1, 12.0,
                  6.1, 4.9, 5.1, 6.3]
}

data = pd.DataFrame(data_dict)
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

pattern_colors = {
    'Stables': '#2E86AB',
    'Croissance': '#06A77D',
    'Oscillatoires': '#F18F01',
    'Décroissance': '#C73E1D'
}

print(f"\n✅ Données chargées: {data['Projet'].nunique()} projets, {len(data)} versions")

# ============================================================================
# FONCTIONS D'EXPORT OPTIMISÉES
# ============================================================================

def save_figure(fig, filename, formats=['png', 'pdf']):
    """
    Sauvegarde une figure dans plusieurs formats optimisés pour Word.
    """
    saved_files = []
    for fmt in formats:
        filepath = f"{filename}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=DPI, bbox_inches='tight',
                   pad_inches=0.1, facecolor='white')
        saved_files.append(filepath)
        print(f"   ✅ {filepath}")
    return saved_files

# ============================================================================
# FIGURE 1: GRAPHIQUE 2×2 PRINCIPAL (SYNTHÈSE DES 4 PATTERNS)
# ============================================================================

print("\n" + "="*70)
print("FIGURE 1: VUE D'ENSEMBLE 2×2 - SYNTHÈSE DES 4 PATTERNS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=GRID_2x2_SIZE)
axes = axes.ravel()

for idx, (pattern_name, projects) in enumerate(patterns.items()):
    ax = axes[idx]

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2, markersize=5,
                label=projet, alpha=0.85)

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
    ax.set_title(f'{pattern_name}', fontsize=10, fontweight='bold',
                 pad=8, color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=False, fancybox=False, fontsize=7)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Ajuster les marges internes
    ax.margins(x=0.05, y=0.05)

plt.suptitle('Patterns d\'évolution temporelle des métriques (LOC moyen)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()

save_figure(fig, 'Fig_A3_1_Vue_Ensemble_4_Patterns')
plt.show()

# ============================================================================
# FIGURE 2: PATTERNS STABLES (DÉTAIL)
# ============================================================================

print("\n" + "="*70)
print("FIGURE 2: PATTERN STABLES (DÉTAIL)")
print("="*70)

fig, ax = plt.subplots(figsize=SINGLE_FIG_SIZE)

for projet in patterns['Stables']:
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
    ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
            marker='o', linewidth=2.5, markersize=6,
            label=projet, alpha=0.85)

ax.set_xlabel('Version (ordre chronologique)', fontsize=10, fontweight='bold')
ax.set_ylabel('LOC moyen (lignes de code)', fontsize=10, fontweight='bold')
ax.set_title('Pattern Stables : Ant, Log4j, POI, Ivy',
             fontsize=11, fontweight='bold', pad=10,
             color=pattern_colors['Stables'])
ax.legend(loc='best', frameon=True, shadow=False, fontsize=8)
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_figure(fig, 'Fig_A3_2_Pattern_Stables')
plt.show()

# ============================================================================
# FIGURE 3: PATTERNS CROISSANCE (DÉTAIL)
# ============================================================================

print("\n" + "="*70)
print("FIGURE 3: PATTERN CROISSANCE (DÉTAIL)")
print("="*70)

fig, ax = plt.subplots(figsize=SINGLE_FIG_SIZE)

for projet in patterns['Croissance']:
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
    ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
            marker='o', linewidth=2.5, markersize=6,
            label=projet, alpha=0.85)

ax.set_xlabel('Version (ordre chronologique)', fontsize=10, fontweight='bold')
ax.set_ylabel('LOC moyen (lignes de code)', fontsize=10, fontweight='bold')
ax.set_title('Pattern Croissance : Lucene, Xalan, Synapse',
             fontsize=11, fontweight='bold', pad=10,
             color=pattern_colors['Croissance'])
ax.legend(loc='best', frameon=True, shadow=False, fontsize=8)
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_figure(fig, 'Fig_A3_3_Pattern_Croissance')
plt.show()

# ============================================================================
# FIGURE 4: PATTERNS OSCILLATOIRES (DÉTAIL)
# ============================================================================

print("\n" + "="*70)
print("FIGURE 4: PATTERN OSCILLATOIRES (DÉTAIL)")
print("="*70)

fig, ax = plt.subplots(figsize=SINGLE_FIG_SIZE)

for projet in patterns['Oscillatoires']:
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
    ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
            marker='o', linewidth=2.5, markersize=6,
            label=projet, alpha=0.85)

ax.set_xlabel('Version (ordre chronologique)', fontsize=10, fontweight='bold')
ax.set_ylabel('LOC moyen (lignes de code)', fontsize=10, fontweight='bold')
ax.set_title('Pattern Oscillatoires : Velocity, Camel',
             fontsize=11, fontweight='bold', pad=10,
             color=pattern_colors['Oscillatoires'])
ax.legend(loc='best', frameon=True, shadow=False, fontsize=8)
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_figure(fig, 'Fig_A3_4_Pattern_Oscillatoires')
plt.show()

# ============================================================================
# FIGURE 5: PATTERNS DÉCROISSANCE (DÉTAIL)
# ============================================================================

print("\n" + "="*70)
print("FIGURE 5: PATTERN DÉCROISSANCE (DÉTAIL)")
print("="*70)

fig, ax = plt.subplots(figsize=SINGLE_FIG_SIZE)

for projet in patterns['Décroissance']:
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
    ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
            marker='o', linewidth=2.5, markersize=6,
            label=projet, alpha=0.85)

ax.set_xlabel('Version (ordre chronologique)', fontsize=10, fontweight='bold')
ax.set_ylabel('LOC moyen (lignes de code)', fontsize=10, fontweight='bold')
ax.set_title('Pattern Décroissance : JEdit, Xerces',
             fontsize=11, fontweight='bold', pad=10,
             color=pattern_colors['Décroissance'])
ax.legend(loc='best', frameon=True, shadow=False, fontsize=8)
ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_figure(fig, 'Fig_A3_5_Pattern_Decroissance')
plt.show()

# ============================================================================
# FIGURE 6: COMPARAISON LOC vs WMC (2×1)
# ============================================================================

print("\n" + "="*70)
print("FIGURE 6: COMPARAISON LOC vs WMC")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=GRID_2x1_SIZE)

# Sous-graphique 1: LOC
ax1 = axes[0]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax1.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax1.set_xlabel('Version', fontsize=9, fontweight='bold')
ax1.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
ax1.set_title('(a) Taille du code (LOC)', fontsize=10, fontweight='bold', pad=8)
ax1.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Sous-graphique 2: WMC
ax2 = axes[1]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax2.plot(projet_data['Version_Num'], projet_data['WMC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax2.set_xlabel('Version', fontsize=9, fontweight='bold')
ax2.set_ylabel('WMC moyen', fontsize=9, fontweight='bold')
ax2.set_title('(b) Complexité (WMC)', fontsize=10, fontweight='bold', pad=8)
ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Légende commune
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Évolution des métriques par pattern',
             fontsize=11, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)

save_figure(fig, 'Fig_A3_6_Comparaison_LOC_WMC')
plt.show()

# ============================================================================
# TABLEAU RÉCAPITULATIF
# ============================================================================

print("\n" + "="*70)
print("TABLEAU RÉCAPITULATIF POUR INTÉGRATION WORD")
print("="*70)

def calculate_trend(projet_data, metric):
    x = np.arange(len(projet_data))
    y = projet_data[metric].values
    if len(x) < 2:
        return 0
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope

summary_data = []
for pattern_name, projects in patterns.items():
    pattern_data = data[data['Projet'].isin(projects)]

    avg_trends = {
        'LOC': np.mean([calculate_trend(data[data['Projet']==p], 'LOC Moyen')
                       for p in projects]),
        'WMC': np.mean([calculate_trend(data[data['Projet']==p], 'WMC Moyen')
                       for p in projects])
    }

    summary_data.append({
        'Pattern': pattern_name,
        'Projets': ', '.join(projects),
        'Nb Versions': len(pattern_data),
        'LOC moyen': f"{pattern_data['LOC Moyen'].mean():.0f}",
        'Tendance LOC': f"{avg_trends['LOC']:+.1f}",
        'WMC moyen': f"{pattern_data['WMC Moyen'].mean():.1f}",
        'Tendance WMC': f"{avg_trends['WMC']:+.2f}"
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

# Export pour Word
df_summary.to_csv('Tableau_A3_Recapitulatif.csv', index=False)
print("\n✅ Tableau sauvegardé: Tableau_A3_Recapitulatif.csv")

# ============================================================================
# RÉSUMÉ DES FICHIERS GÉNÉRÉS
# ============================================================================

print("\n" + "="*70)
print("FICHIERS GÉNÉRÉS POUR INTÉGRATION WORD")
print("="*70)

print("\n📊 FIGURES (PNG + PDF, 300 DPI):")
print("   1. Fig_A3_1_Vue_Ensemble_4_Patterns (6.5\" × 8\")")
print("   2. Fig_A3_2_Pattern_Stables (6.5\" × 4\")")
print("   3. Fig_A3_3_Pattern_Croissance (6.5\" × 4\")")
print("   4. Fig_A3_4_Pattern_Oscillatoires (6.5\" × 4\")")
print("   5. Fig_A3_5_Pattern_Decroissance (6.5\" × 4\")")
print("   6. Fig_A3_6_Comparaison_LOC_WMC (6.5\" × 6\")")

print("\n📁 TABLEAU:")
print("   - Tableau_A3_Recapitulatif.csv")

print("\n" + "="*70)
print("INSTRUCTIONS D'INTÉGRATION DANS WORD")
print("="*70)

instructions = """
1. INSÉRER LES FIGURES:
   - Dans Word: Insertion > Images > Cet appareil
   - Sélectionner les fichiers PNG (meilleure compatibilité Word)
   - Taille: Garder "6.5 pouces" de largeur (largeur pleine page)

2. LÉGENDES SUGGÉRÉES:

   Figure A.3.1: Synthèse des quatre patterns d'évolution temporelle
   identifiés dans le corpus PROMISE. (a) Stables: métriques constantes
   justifiant la standardisation globale. (b) Croissance: tendances
   croissantes justifiant la standardisation par version. (c) Oscillatoires:
   variations périodiques justifiant l'approche hybride. (d) Décroissance:
   amélioration continue renforçant l'importance de la chronologie temporelle.

   Figure A.3.2: Pattern Stables (Ant, Log4j, POI, Ivy). Métriques
   relativement constantes entre versions suggérant l'efficacité de la
   stratégie de standardisation globale (SG).

   Figure A.3.3: Pattern Croissance (Lucene, Xalan, Synapse). Tendances
   croissantes systématiques justifiant la stratégie par version (SV).

   Figure A.3.4: Pattern Oscillatoires (Velocity, Camel). Variations
   périodiques motivant la stratégie hybride SR-Robuste.

   Figure A.3.5: Pattern Décroissance (JEdit, Xerces). Efforts d'amélioration
   continue (refactoring) renforçant l'importance du respect de la chronologie.

   Figure A.3.6: Comparaison de l'évolution de la taille du code (LOC) et
   de la complexité (WMC) pour les quatre patterns identifiés.

3. QUALITÉ D'IMPRESSION:
   - Les figures sont à 300 DPI (excellente qualité pour impression)
   - Format PNG: meilleure compatibilité Word
   - Format PDF: alternative si problèmes d'affichage

4. MISE EN PAGE:
   - Les figures s'adaptent automatiquement à la largeur de page
   - Marges respectées (2.5 cm)
   - Texte lisible même en format réduit
"""

print(instructions)

print("\n✅ GÉNÉRATION TERMINÉE - PRÊT POUR WORD!")
print("="*70)

In [ ]:
"""
================================================================================
ANNEXE A.3 - PATTERNS D'ÉVOLUTION TEMPORELLE - VERSION WORD COMPLÈTE
================================================================================

Version optimisée pour intégration dans document Word
INCLUANT toutes les visualisations importantes :
- Vue d'ensemble 2×2
- Détails par pattern (LOC)
- Comparaison LOC vs WMC
- Heatmap des variations relatives
- Évolution normalisée (tous projets)

Tailles adaptées au format A4 avec marges 2.5cm
Résolution 300 DPI pour impression de qualité

Auteure: Mireille Kafando
Institution: UQTR
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION OPTIMISÉE POUR WORD
# ============================================================================

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

# Tailles optimisées pour Word (largeur max: 6.5")
SINGLE_FIG = (6.5, 4)      # Graphique standard
TALL_FIG = (6.5, 5)        # Graphique plus haut
GRID_2x2 = (6.5, 8)        # Grille 2×2
GRID_2x1 = (6.5, 6)        # Grille 2×1
HEATMAP_FIG = (6.5, 4.5)   # Heatmap
NORMALIZED_FIG = (6.5, 8)  # Évolution normalisée
DPI = 300

# Configuration polices
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

print("✅ Configuration Word optimisée chargée")
print(f"   - Résolution: {DPI} DPI")
print(f"   - Largeur max: 6.5 pouces")
print(f"   - Formats: PNG + PDF")

# ============================================================================
# DONNÉES
# ============================================================================

data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'Version': ['ant-1.3', 'ant-1.4', 'ant-1.5', 'ant-1.6', 'ant-1.7',
                'camel-1.0', 'camel-1.2', 'camel-1.4', 'camel-1.6',
                'ivy-1.0', 'ivy-1.1', 'ivy-1.2',
                'jedit-3.2', 'jedit-4.0', 'jedit-4.1', 'jedit-4.2', 'jedit-4.3',
                'log4j-1.0', 'log4j-1.1', 'log4j-1.2',
                'lucene-2.0', 'lucene-2.2', 'lucene-2.4',
                'poi-1.5', 'poi-2.0', 'poi-2.5', 'poi-3.0',
                'synapse-1.0', 'synapse-1.1', 'synapse-1.2',
                'velocity-1.4', 'velocity-1.5', 'velocity-1.6',
                'xalan-2.4', 'xalan-2.5', 'xalan-2.6', 'xalan-2.7',
                'xerces-1.1', 'xerces-1.2', 'xerces-1.3', 'xerces-1.4.4'],
    'LOC Moyen': [302, 304, 297, 323, 280, 99, 109, 112, 117, 246, 246, 249,
                  474, 473, 491, 465, 411, 160, 183, 186, 259, 257, 303,
                  234, 297, 311, 293, 183, 191, 209, 264, 248, 249,
                  311, 380, 465, 471, 560, 362, 369, 240],
    'WMC Moyen': [10.6, 10.5, 10.1, 11.1, 11.1, 8.1, 8.3, 8.5, 8.6,
                  10.6, 10.7, 11.3, 12.5, 12.9, 13.1, 13.2, 12.4,
                  6.6, 7.8, 8.4, 9.3, 9.4, 10.4, 13.4, 14.3, 14.3, 13.5,
                  8.1, 7.3, 7.6, 8.8, 9.0, 9.0, 11.4, 11.3, 11.0, 10.9,
                  11.4, 11.3, 11.4, 9.9],
    'Classes': [125, 178, 293, 351, 745, 339, 608, 872, 965, 111, 241, 352,
                272, 306, 312, 367, 492, 135, 109, 205, 195, 247, 340,
                237, 314, 385, 442, 157, 222, 256, 196, 214, 229,
                723, 803, 885, 909, 162, 440, 453, 588],
    'CBO Moyen': [10.4, 10.8, 10.6, 11.5, 11.0, 10.0, 10.1, 10.8, 11.1,
                  10.4, 11.3, 13.2, 12.0, 12.4, 13.0, 14.1, 14.3,
                  6.9, 6.9, 7.6, 9.8, 10.0, 10.8, 8.3, 8.7, 9.0, 10.1,
                  13.2, 12.5, 12.8, 10.6, 10.6, 10.8, 14.5, 12.9, 12.1, 12.0,
                  6.1, 4.9, 5.1, 6.3]
}

data = pd.DataFrame(data_dict)
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

pattern_colors = {
    'Stables': '#2E86AB',
    'Croissance': '#06A77D',
    'Oscillatoires': '#F18F01',
    'Décroissance': '#C73E1D'
}

print(f"\n✅ Données chargées: {data['Projet'].nunique()} projets, {len(data)} versions")

# ============================================================================
# FONCTION D'EXPORT
# ============================================================================

def save_fig(fig, filename):
    """Sauvegarde en PNG et PDF"""
    for fmt in ['png', 'pdf']:
        filepath = f"{filename}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=DPI, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"   ✅ {filepath}")

# ============================================================================
# FIGURE 1: VUE D'ENSEMBLE 2×2
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.1: VUE D'ENSEMBLE 2×2 - SYNTHÈSE DES 4 PATTERNS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=GRID_2x2)
axes = axes.ravel()

for idx, (pattern_name, projects) in enumerate(patterns.items()):
    ax = axes[idx]

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2, markersize=5, label=projet, alpha=0.85)

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
    ax.set_title(f'{pattern_name}', fontsize=10, fontweight='bold',
                 pad=8, color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=False, fontsize=7)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Patterns d\'évolution temporelle des métriques (LOC moyen)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()
save_fig(fig, 'Fig_A3_1_Vue_Ensemble_4_Patterns')
plt.show()

# ============================================================================
# FIGURES 2-5: DÉTAILS PAR PATTERN (LOC)
# ============================================================================

print("\n" + "="*70)
print("FIGURES A.3.2 à A.3.5: DÉTAILS PAR PATTERN")
print("="*70)

for idx, (pattern_name, projects) in enumerate(patterns.items(), start=2):
    fig, ax = plt.subplots(figsize=SINGLE_FIG)

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2.5, markersize=6, label=projet, alpha=0.85)

    ax.set_xlabel('Version (ordre chronologique)', fontsize=10, fontweight='bold')
    ax.set_ylabel('LOC moyen (lignes de code)', fontsize=10, fontweight='bold')
    ax.set_title(f'Pattern {pattern_name}', fontsize=11, fontweight='bold',
                 pad=10, color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=False, fontsize=8)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    filename = f'Fig_A3_{idx}_Pattern_{pattern_name.replace(" ", "_")}'
    save_fig(fig, filename)
    plt.show()

# ============================================================================
# FIGURE 6: COMPARAISON LOC vs WMC (2×1)
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.6: COMPARAISON LOC vs WMC")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=GRID_2x1)

# LOC
ax1 = axes[0]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax1.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax1.set_xlabel('Version', fontsize=9, fontweight='bold')
ax1.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
ax1.set_title('(a) Taille du code (LOC)', fontsize=10, fontweight='bold', pad=8)
ax1.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# WMC
ax2 = axes[1]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax2.plot(projet_data['Version_Num'], projet_data['WMC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax2.set_xlabel('Version', fontsize=9, fontweight='bold')
ax2.set_ylabel('WMC moyen', fontsize=9, fontweight='bold')
ax2.set_title('(b) Complexité (WMC)', fontsize=10, fontweight='bold', pad=8)
ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Comparaison LOC vs WMC par pattern',
             fontsize=11, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
save_fig(fig, 'Fig_A3_6_Comparaison_LOC_WMC')
plt.show()

# ============================================================================
# FIGURE 7: HEATMAP DES VARIATIONS RELATIVES
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.7: HEATMAP DES VARIATIONS RELATIVES")
print("="*70)

variations = []
for projet in data['Projet'].unique():
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

    if len(projet_data) > 1:
        var_loc = ((projet_data.iloc[-1]['LOC Moyen'] - projet_data.iloc[0]['LOC Moyen'])
                   / projet_data.iloc[0]['LOC Moyen'] * 100)
        var_wmc = ((projet_data.iloc[-1]['WMC Moyen'] - projet_data.iloc[0]['WMC Moyen'])
                   / projet_data.iloc[0]['WMC Moyen'] * 100)
        var_classes = ((projet_data.iloc[-1]['Classes'] - projet_data.iloc[0]['Classes'])
                       / projet_data.iloc[0]['Classes'] * 100)
        var_cbo = ((projet_data.iloc[-1]['CBO Moyen'] - projet_data.iloc[0]['CBO Moyen'])
                   / projet_data.iloc[0]['CBO Moyen'] * 100)

        variations.append({
            'Projet': projet,
            'LOC': var_loc,
            'WMC': var_wmc,
            'Classes': var_classes,
            'CBO': var_cbo
        })

df_variations = pd.DataFrame(variations)
df_variations = df_variations.set_index('Projet')

fig, ax = plt.subplots(figsize=HEATMAP_FIG)

sns.heatmap(df_variations.T, annot=True, fmt='.0f', cmap='RdYlGn_r',
            center=0, cbar_kws={'label': 'Variation (%)', 'shrink': 0.8},
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 8})

ax.set_title('Variations relatives des métriques\n(première vs dernière version)',
             fontweight='bold', pad=12, fontsize=11)
ax.set_xlabel('Projets', fontweight='bold', fontsize=10)
ax.set_ylabel('Métriques', fontweight='bold', fontsize=10)

plt.tight_layout()
save_fig(fig, 'Fig_A3_7_Heatmap_Variations')
plt.show()

# Sauvegarder aussi en CSV
df_variations.to_csv('Tableau_A3_Variations_Detaillees.csv')
print("   ✅ Tableau_A3_Variations_Detaillees.csv")

# ============================================================================
# FIGURE 8: ÉVOLUTION NORMALISÉE (TOUS PROJETS, 4 MÉTRIQUES)
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.8: ÉVOLUTION NORMALISÉE - TOUS PROJETS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=NORMALIZED_FIG)

metrics_to_plot = ['LOC Moyen', 'WMC Moyen', 'Classes', 'CBO Moyen']
metric_labels = ['LOC moyen', 'WMC moyen', 'Nombre de classes', 'CBO moyen']

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[idx // 2, idx % 2]

    for pattern_name, projects in patterns.items():
        for projet in projects:
            projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

            # Normalisation min-max
            values = projet_data[metric].values
            if values.max() > values.min():
                normalized = (values - values.min()) / (values.max() - values.min())
            else:
                normalized = values * 0  # Si pas de variation, mettre à 0

            ax.plot(projet_data['Version_Num'], normalized,
                    marker='o', linewidth=1.5, markersize=4,
                    alpha=0.6, color=pattern_colors[pattern_name])

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('Valeur normalisée (0-1)', fontsize=9, fontweight='bold')
    ax.set_title(f'({chr(97+idx)}) {label}', fontsize=10, fontweight='bold', pad=8)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-0.05, 1.05)

# Légende commune
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Évolution normalisée des métriques par pattern\n' +
             '(valeurs normalisées 0-1 pour comparaison)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
save_fig(fig, 'Fig_A3_8_Evolution_Normalisee')
plt.show()

# ============================================================================
# TABLEAU RÉCAPITULATIF
# ============================================================================

print("\n" + "="*70)
print("TABLEAU RÉCAPITULATIF")
print("="*70)

def calculate_trend(projet_data, metric):
    x = np.arange(len(projet_data))
    y = projet_data[metric].values
    if len(x) < 2:
        return 0
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope

summary_data = []
for pattern_name, projects in patterns.items():
    pattern_data = data[data['Projet'].isin(projects)]

    avg_trends = {
        'LOC': np.mean([calculate_trend(data[data['Projet']==p], 'LOC Moyen')
                       for p in projects]),
        'WMC': np.mean([calculate_trend(data[data['Projet']==p], 'WMC Moyen')
                       for p in projects])
    }

    summary_data.append({
        'Pattern': pattern_name,
        'Projets': ', '.join(projects),
        'Nb Versions': len(pattern_data),
        'LOC moyen': f"{pattern_data['LOC Moyen'].mean():.0f}",
        'Tendance LOC': f"{avg_trends['LOC']:+.1f}",
        'WMC moyen': f"{pattern_data['WMC Moyen'].mean():.1f}",
        'Tendance WMC': f"{avg_trends['WMC']:+.2f}",
        'Stratégie': ['SG (globale)', 'SV (par version)',
                      'SR-Robuste', 'Chronologie'][list(patterns.keys()).index(pattern_name)]
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

df_summary.to_csv('Tableau_A3_Recapitulatif.csv', index=False)
print("\n✅ Tableau_A3_Recapitulatif.csv")

# ============================================================================
# RÉSUMÉ FINAL
# ============================================================================

print("\n" + "="*70)
print("✅ GÉNÉRATION TERMINÉE - VERSION WORD COMPLÈTE")
print("="*70)

print("\n📊 FICHIERS GÉNÉRÉS (PNG + PDF, 300 DPI):")
print("\n   FIGURES PRINCIPALES:")
print("   1. Fig_A3_1_Vue_Ensemble_4_Patterns (6.5\" × 8\")")
print("   2. Fig_A3_2_Pattern_Stables (6.5\" × 4\")")
print("   3. Fig_A3_3_Pattern_Croissance (6.5\" × 4\")")
print("   4. Fig_A3_4_Pattern_Oscillatoires (6.5\" × 4\")")
print("   5. Fig_A3_5_Pattern_Décroissance (6.5\" × 4\")")
print("   6. Fig_A3_6_Comparaison_LOC_WMC (6.5\" × 6\")")
print("   7. Fig_A3_7_Heatmap_Variations (6.5\" × 4.5\") ⭐ NOUVEAU")
print("   8. Fig_A3_8_Evolution_Normalisee (6.5\" × 8\") ⭐ NOUVEAU")

print("\n   TABLEAUX CSV:")
print("   - Tableau_A3_Recapitulatif.csv")
print("   - Tableau_A3_Variations_Detaillees.csv ⭐ NOUVEAU")

print("\n✨ Total: 8 figures + 2 tableaux")
print("🎯 Toutes les tailles optimisées pour Word (6.5\" large max)")
print("📄 Résolution professionnelle (300 DPI)")
print("="*70)

# ============================================================================
# GUIDE D'INTÉGRATION
# ============================================================================

print("\n" + "="*70)
print("GUIDE D'INTÉGRATION DANS WORD")
print("="*70)

guide = """
📝 STRUCTURE RECOMMANDÉE ANNEXE A.3

ANNEXE A.3 : PATTERNS D'ÉVOLUTION TEMPORELLE DES MÉTRIQUES

Introduction (1 paragraphe)
Notre analyse qualitative révèle quatre patterns distincts...

[Figure A.3.1 - Vue d'ensemble 2×2] ← GRAPHIQUE PRINCIPAL

A.3.1 Pattern Stables
Description des projets Ant, Log4j, POI, Ivy (1 paragraphe)
[Figure A.3.2 - Pattern Stables]

A.3.2 Pattern Croissance
Description des projets Lucene, Xalan, Synapse (1 paragraphe)
[Figure A.3.3 - Pattern Croissance]

A.3.3 Pattern Oscillatoires
Description des projets Velocity, Camel (1 paragraphe)
[Figure A.3.4 - Pattern Oscillatoires]

A.3.4 Pattern Décroissance
Description des projets JEdit, Xerces (1 paragraphe)
[Figure A.3.5 - Pattern Décroissance]

A.3.5 Comparaison des métriques
Analyse comparative LOC vs WMC (1 paragraphe)
[Figure A.3.6 - Comparaison LOC vs WMC]

A.3.6 Analyse des variations
Synthèse des variations relatives (1 paragraphe)
[Figure A.3.7 - Heatmap Variations] ⭐
[Tableau A.3 - Variations détaillées]

A.3.7 Évolution normalisée
Comparaison à travers toutes les métriques (1 paragraphe)
[Figure A.3.8 - Évolution normalisée] ⭐

Conclusion (1 paragraphe)
Synthèse des implications pour les stratégies de prétraitement...

📊 LÉGENDES SUGGÉRÉES

Figure A.3.7 – Heatmap des variations relatives (%) des métriques entre
première et dernière version. Les valeurs positives (teintes rouges) indiquent
une augmentation, les valeurs négatives (teintes vertes) une diminution. Les
projets en croissance (Lucene, Xalan, Synapse) montrent des augmentations
coordonnées de LOC et Classes, tandis que les projets en décroissance (JEdit,
Xerces) présentent des patterns hétérogènes suggérant des efforts de
refactoring ciblés.

Figure A.3.8 – Évolution normalisée des quatre métriques principales (LOC,
WMC, Classes, CBO) pour l'ensemble des projets. Les valeurs sont normalisées
entre 0 et 1 pour permettre la comparaison. Les patterns de croissance montrent
une évolution monotone croissante, les patterns stables restent dans une bande
étroite, les patterns oscillatoires exhibent des fluctuations périodiques, et
les patterns de décroissance montrent une tendance décroissante globale,
particulièrement marquée pour LOC et Classes.
"""

print(guide)

print("\n✅ PRÊT POUR INTÉGRATION DANS VOTRE MÉMOIRE UQTR !")
print("="*70)

In [ ]:
"""
================================================================================
ANNEXE A.3 - PATTERNS D'ÉVOLUTION TEMPORELLE - VERSION WORD COMPLÈTE
================================================================================

Version optimisée pour intégration dans document Word
INCLUANT toutes les visualisations importantes :
- Vue d'ensemble 2×2
- Détails par pattern (LOC)
- Comparaison LOC vs WMC
- Heatmap des variations relatives
- Évolution normalisée (tous projets)

Tailles adaptées au format A4 avec marges 2.5cm
Résolution 300 DPI pour impression de qualité

Auteure: Mireille Kafando
Institution: UQTR
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION OPTIMISÉE POUR WORD
# ============================================================================

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

# Tailles optimisées pour Word (largeur max: 6.5")
SINGLE_FIG = (6.5, 4)      # Graphique standard
TALL_FIG = (6.5, 5)        # Graphique plus haut
GRID_2x2 = (6.5, 8)        # Grille 2×2
GRID_2x1 = (6.5, 6)        # Grille 2×1
HEATMAP_FIG = (6.5, 4.5)   # Heatmap
NORMALIZED_FIG = (6.5, 8)  # Évolution normalisée
DPI = 300

# Configuration polices
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

print("✅ Configuration Word optimisée chargée")
print(f"   - Résolution: {DPI} DPI")
print(f"   - Largeur max: 6.5 pouces")
print(f"   - Formats: PNG + PDF")

# ============================================================================
# DONNÉES
# ============================================================================

data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'Version': ['ant-1.3', 'ant-1.4', 'ant-1.5', 'ant-1.6', 'ant-1.7',
                'camel-1.0', 'camel-1.2', 'camel-1.4', 'camel-1.6',
                'ivy-1.0', 'ivy-1.1', 'ivy-1.2',
                'jedit-3.2', 'jedit-4.0', 'jedit-4.1', 'jedit-4.2', 'jedit-4.3',
                'log4j-1.0', 'log4j-1.1', 'log4j-1.2',
                'lucene-2.0', 'lucene-2.2', 'lucene-2.4',
                'poi-1.5', 'poi-2.0', 'poi-2.5', 'poi-3.0',
                'synapse-1.0', 'synapse-1.1', 'synapse-1.2',
                'velocity-1.4', 'velocity-1.5', 'velocity-1.6',
                'xalan-2.4', 'xalan-2.5', 'xalan-2.6', 'xalan-2.7',
                'xerces-1.1', 'xerces-1.2', 'xerces-1.3', 'xerces-1.4.4'],
    'LOC Moyen': [302, 304, 297, 323, 280, 99, 109, 112, 117, 246, 246, 249,
                  474, 473, 491, 465, 411, 160, 183, 186, 259, 257, 303,
                  234, 297, 311, 293, 183, 191, 209, 264, 248, 249,
                  311, 380, 465, 471, 560, 362, 369, 240],
    'WMC Moyen': [10.6, 10.5, 10.1, 11.1, 11.1, 8.1, 8.3, 8.5, 8.6,
                  10.6, 10.7, 11.3, 12.5, 12.9, 13.1, 13.2, 12.4,
                  6.6, 7.8, 8.4, 9.3, 9.4, 10.4, 13.4, 14.3, 14.3, 13.5,
                  8.1, 7.3, 7.6, 8.8, 9.0, 9.0, 11.4, 11.3, 11.0, 10.9,
                  11.4, 11.3, 11.4, 9.9],
    'NOC': [125, 178, 293, 351, 745, 339, 608, 872, 965, 111, 241, 352,
                272, 306, 312, 367, 492, 135, 109, 205, 195, 247, 340,
                237, 314, 385, 442, 157, 222, 256, 196, 214, 229,
                723, 803, 885, 909, 162, 440, 453, 588],
    'CBO Moyen': [10.4, 10.8, 10.6, 11.5, 11.0, 10.0, 10.1, 10.8, 11.1,
                  10.4, 11.3, 13.2, 12.0, 12.4, 13.0, 14.1, 14.3,
                  6.9, 6.9, 7.6, 9.8, 10.0, 10.8, 8.3, 8.7, 9.0, 10.1,
                  13.2, 12.5, 12.8, 10.6, 10.6, 10.8, 14.5, 12.9, 12.1, 12.0,
                  6.1, 4.9, 5.1, 6.3]
}

data = pd.DataFrame(data_dict)
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

pattern_colors = {
    'Stables': '#2E86AB',
    'Croissance': '#06A77D',
    'Oscillatoires': '#F18F01',
    'Décroissance': '#C73E1D'
}

print(f"\n✅ Données chargées: {data['Projet'].nunique()} projets, {len(data)} versions")

# ============================================================================
# FONCTION D'EXPORT
# ============================================================================

def save_fig(fig, filename):
    """Sauvegarde en PNG et PDF"""
    for fmt in ['png', 'pdf']:
        filepath = f"{filename}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=DPI, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"   ✅ {filepath}")

# ============================================================================
# FIGURE 1: VUE D'ENSEMBLE 2×2
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.1: VUE D'ENSEMBLE 2×2 - SYNTHÈSE DES 4 PATTERNS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=GRID_2x2)
axes = axes.ravel()

for idx, (pattern_name, projects) in enumerate(patterns.items()):
    ax = axes[idx]

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2, markersize=5, label=projet, alpha=0.85)

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
    ax.set_title(f'{pattern_name}', fontsize=10, fontweight='bold',
                 pad=8, color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=False, fontsize=7)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Patterns d\'évolution temporelle des métriques (LOC moyen)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()
save_fig(fig, 'Fig_A3_1_Vue_Ensemble_4_Patterns')
plt.show()

# ============================================================================
# FIGURES 2-5: DÉTAILS PAR PATTERN (LOC)
# ============================================================================

print("\n" + "="*70)
print("FIGURES A.3.2 à A.3.5: DÉTAILS PAR PATTERN")
print("="*70)

for idx, (pattern_name, projects) in enumerate(patterns.items(), start=2):
    fig, ax = plt.subplots(figsize=SINGLE_FIG)

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2.5, markersize=6, label=projet, alpha=0.85)

    ax.set_xlabel('Version (ordre chronologique)', fontsize=10, fontweight='bold')
    ax.set_ylabel('LOC moyen (lignes de code)', fontsize=10, fontweight='bold')
    ax.set_title(f'Pattern {pattern_name}', fontsize=11, fontweight='bold',
                 pad=10, color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=False, fontsize=8)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    filename = f'Fig_A3_{idx}_Pattern_{pattern_name.replace(" ", "_")}'
    save_fig(fig, filename)
    plt.show()

# ============================================================================
# FIGURE 6: COMPARAISON LOC vs WMC (2×1)
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.6: COMPARAISON LOC vs WMC")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=GRID_2x1)

# LOC
ax1 = axes[0]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax1.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax1.set_xlabel('Version', fontsize=9, fontweight='bold')
ax1.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
ax1.set_title('(a) Taille du code (LOC)', fontsize=10, fontweight='bold', pad=8)
ax1.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# WMC
ax2 = axes[1]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax2.plot(projet_data['Version_Num'], projet_data['WMC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax2.set_xlabel('Version', fontsize=9, fontweight='bold')
ax2.set_ylabel('WMC moyen', fontsize=9, fontweight='bold')
ax2.set_title('(b) Complexité (WMC)', fontsize=10, fontweight='bold', pad=8)
ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Comparaison LOC vs WMC par pattern',
             fontsize=11, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
save_fig(fig, 'Fig_A3_6_Comparaison_LOC_WMC')
plt.show()

# ============================================================================
# FIGURE 7: HEATMAP DES VARIATIONS RELATIVES
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.7: HEATMAP DES VARIATIONS RELATIVES")
print("="*70)

variations = []
for projet in data['Projet'].unique():
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

    if len(projet_data) > 1:
        var_loc = ((projet_data.iloc[-1]['LOC Moyen'] - projet_data.iloc[0]['LOC Moyen'])
                   / projet_data.iloc[0]['LOC Moyen'] * 100)
        var_wmc = ((projet_data.iloc[-1]['WMC Moyen'] - projet_data.iloc[0]['WMC Moyen'])
                   / projet_data.iloc[0]['WMC Moyen'] * 100)
        var_classes = ((projet_data.iloc[-1]['NOC'] - projet_data.iloc[0]['NOC'])
                       / projet_data.iloc[0]['NOC'] * 100)
        var_cbo = ((projet_data.iloc[-1]['CBO Moyen'] - projet_data.iloc[0]['CBO Moyen'])
                   / projet_data.iloc[0]['CBO Moyen'] * 100)

        variations.append({
            'Projet': projet,
            'LOC': var_loc,
            'WMC': var_wmc,
            'NOC': var_classes,
            'CBO': var_cbo
        })

df_variations = pd.DataFrame(variations)
df_variations = df_variations.set_index('Projet')

fig, ax = plt.subplots(figsize=HEATMAP_FIG)

sns.heatmap(df_variations.T, annot=True, fmt='.0f', cmap='RdYlGn_r',
            center=0, cbar_kws={'label': 'Variation (%)', 'shrink': 0.8},
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 8})

ax.set_title('Variations relatives des métriques\n(première vs dernière version)',
             fontweight='bold', pad=12, fontsize=11)
ax.set_xlabel('Projets', fontweight='bold', fontsize=10)
ax.set_ylabel('Métriques', fontweight='bold', fontsize=10)

plt.tight_layout()
save_fig(fig, 'Fig_A3_7_Heatmap_Variations')
plt.show()

# Sauvegarder aussi en CSV
df_variations.to_csv('Tableau_A3_Variations_Detaillees.csv')
print("   ✅ Tableau_A3_Variations_Detaillees.csv")

# ============================================================================
# FIGURE 8: ÉVOLUTION NORMALISÉE (TOUS PROJETS, 4 MÉTRIQUES)
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.8: ÉVOLUTION NORMALISÉE - TOUS PROJETS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=NORMALIZED_FIG)

metrics_to_plot = ['LOC Moyen', 'WMC Moyen', 'NOC', 'CBO Moyen']
metric_labels = ['LOC moyen', 'WMC moyen', 'NOC', 'CBO moyen']

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[idx // 2, idx % 2]

    for pattern_name, projects in patterns.items():
        for projet in projects:
            projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

            # Normalisation min-max
            values = projet_data[metric].values
            if values.max() > values.min():
                normalized = (values - values.min()) / (values.max() - values.min())
            else:
                normalized = values * 0  # Si pas de variation, mettre à 0

            ax.plot(projet_data['Version_Num'], normalized,
                    marker='o', linewidth=1.5, markersize=4,
                    alpha=0.6, color=pattern_colors[pattern_name])

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('Valeur normalisée (0-1)', fontsize=9, fontweight='bold')
    ax.set_title(f'({chr(97+idx)}) {label}', fontsize=10, fontweight='bold', pad=8)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-0.05, 1.05)

# Légende commune
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Évolution normalisée des métriques par pattern\n' +
             '(valeurs normalisées 0-1 pour comparaison)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
save_fig(fig, 'Fig_A3_8_Evolution_Normalisee')
plt.show()

# ============================================================================
# TABLEAU RÉCAPITULATIF
# ============================================================================

print("\n" + "="*70)
print("TABLEAU RÉCAPITULATIF")
print("="*70)

def calculate_trend(projet_data, metric):
    x = np.arange(len(projet_data))
    y = projet_data[metric].values
    if len(x) < 2:
        return 0
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope

summary_data = []
for pattern_name, projects in patterns.items():
    pattern_data = data[data['Projet'].isin(projects)]

    avg_trends = {
        'LOC': np.mean([calculate_trend(data[data['Projet']==p], 'LOC Moyen')
                       for p in projects]),
        'WMC': np.mean([calculate_trend(data[data['Projet']==p], 'WMC Moyen')
                       for p in projects])
    }

    summary_data.append({
        'Pattern': pattern_name,
        'Projets': ', '.join(projects),
        'Nb Versions': len(pattern_data),
        'LOC moyen': f"{pattern_data['LOC Moyen'].mean():.0f}",
        'Tendance LOC': f"{avg_trends['LOC']:+.1f}",
        'WMC moyen': f"{pattern_data['WMC Moyen'].mean():.1f}",
        'Tendance WMC': f"{avg_trends['WMC']:+.2f}",
        'Stratégie': ['SG (globale)', 'SV (par version)',
                      'SR-Robuste', 'Chronologie'][list(patterns.keys()).index(pattern_name)]
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

df_summary.to_csv('Tableau_A3_Recapitulatif.csv', index=False)
print("\n✅ Tableau_A3_Recapitulatif.csv")

# ============================================================================
# RÉSUMÉ FINAL
# ============================================================================

print("\n" + "="*70)
print("✅ GÉNÉRATION TERMINÉE - VERSION WORD COMPLÈTE")
print("="*70)

print("\n📊 FICHIERS GÉNÉRÉS (PNG + PDF, 300 DPI):")
print("\n   FIGURES PRINCIPALES:")
print("   1. Fig_A3_1_Vue_Ensemble_4_Patterns (6.5\" × 8\")")
print("   2. Fig_A3_2_Pattern_Stables (6.5\" × 4\")")
print("   3. Fig_A3_3_Pattern_Croissance (6.5\" × 4\")")
print("   4. Fig_A3_4_Pattern_Oscillatoires (6.5\" × 4\")")
print("   5. Fig_A3_5_Pattern_Décroissance (6.5\" × 4\")")
print("   6. Fig_A3_6_Comparaison_LOC_WMC (6.5\" × 6\")")
print("   7. Fig_A3_7_Heatmap_Variations (6.5\" × 4.5\") ⭐ NOUVEAU")
print("   8. Fig_A3_8_Evolution_Normalisee (6.5\" × 8\") ⭐ NOUVEAU")

print("\n   TABLEAUX CSV:")
print("   - Tableau_A3_Recapitulatif.csv")
print("   - Tableau_A3_Variations_Detaillees.csv ⭐ NOUVEAU")

print("\n✨ Total: 8 figures + 2 tableaux")
print("🎯 Toutes les tailles optimisées pour Word (6.5\" large max)")
print("📄 Résolution professionnelle (300 DPI)")
print("="*70)

# ============================================================================
# GUIDE D'INTÉGRATION
# ============================================================================

print("\n" + "="*70)
print("GUIDE D'INTÉGRATION DANS WORD")
print("="*70)

guide = """
📝 STRUCTURE RECOMMANDÉE ANNEXE A.3

ANNEXE A.3 : PATTERNS D'ÉVOLUTION TEMPORELLE DES MÉTRIQUES

Introduction (1 paragraphe)
Notre analyse qualitative révèle quatre patterns distincts...

[Figure A.3.1 - Vue d'ensemble 2×2] ← GRAPHIQUE PRINCIPAL

A.3.1 Pattern Stables
Description des projets Ant, Log4j, POI, Ivy (1 paragraphe)
[Figure A.3.2 - Pattern Stables]

A.3.2 Pattern Croissance
Description des projets Lucene, Xalan, Synapse (1 paragraphe)
[Figure A.3.3 - Pattern Croissance]

A.3.3 Pattern Oscillatoires
Description des projets Velocity, Camel (1 paragraphe)
[Figure A.3.4 - Pattern Oscillatoires]

A.3.4 Pattern Décroissance
Description des projets JEdit, Xerces (1 paragraphe)
[Figure A.3.5 - Pattern Décroissance]

A.3.5 Comparaison des métriques
Analyse comparative LOC vs WMC (1 paragraphe)
[Figure A.3.6 - Comparaison LOC vs WMC]

A.3.6 Analyse des variations
Synthèse des variations relatives (1 paragraphe)
[Figure A.3.7 - Heatmap Variations] ⭐
[Tableau A.3 - Variations détaillées]

A.3.7 Évolution normalisée
Comparaison à travers toutes les métriques (1 paragraphe)
[Figure A.3.8 - Évolution normalisée] ⭐

Conclusion (1 paragraphe)
Synthèse des implications pour les stratégies de prétraitement...

📊 LÉGENDES SUGGÉRÉES

Figure A.3.7 – Heatmap des variations relatives (%) des métriques entre
première et dernière version. Les valeurs positives (teintes rouges) indiquent
une augmentation, les valeurs négatives (teintes vertes) une diminution. Les
projets en croissance (Lucene, Xalan, Synapse) montrent des augmentations
coordonnées de LOC et NOC, tandis que les projets en décroissance (JEdit,
Xerces) présentent des patterns hétérogènes suggérant des efforts de
refactoring ciblés.

Figure A.3.8 – Évolution normalisée des quatre métriques principales (LOC,
WMC, NOC, CBO) pour l'ensemble des projets. Les valeurs sont normalisées
entre 0 et 1 pour permettre la comparaison. Les patterns de croissance montrent
une évolution monotone croissante, les patterns stables restent dans une bande
étroite, les patterns oscillatoires exhibent des fluctuations périodiques, et
les patterns de décroissance montrent une tendance décroissante globale,
particulièrement marquée pour LOC et NOC.
"""

print(guide)

print("\n✅ PRÊT POUR INTÉGRATION DANS VOTRE MÉMOIRE UQTR !")
print("="*70)

In [ ]:
"""
================================================================================
ANNEXE A.3 - ANALYSES COMPLÉMENTAIRES (VERSION OPTIMALE)
================================================================================

Génère les 3 figures essentielles pour l'Annexe A.3 :
1. Comparaison LOC vs WMC (validation sur 2 métriques)
2. Heatmap des variations (quantification précise)
3. Évolution normalisée 4 métriques (validation robustesse)

Note : La vue d'ensemble 2×2 est maintenant Figure 4.X dans le texte principal
Les détails par pattern (anciennes A.3.2-5) sont superflus car synthétisés
dans Figure 4.X

Tailles optimisées pour Word
Résolution 300 DPI

Auteure: Mireille Kafando
Institution: UQTR
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

# Tailles optimisées
GRID_2x1 = (6.5, 6)
HEATMAP_FIG = (6.5, 4.5)
NORMALIZED_FIG = (6.5, 8)
DPI = 300

plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

print("✅ Configuration chargée")

# ============================================================================
# DONNÉES
# ============================================================================

data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'Version': ['ant-1.3', 'ant-1.4', 'ant-1.5', 'ant-1.6', 'ant-1.7',
                'camel-1.0', 'camel-1.2', 'camel-1.4', 'camel-1.6',
                'ivy-1.0', 'ivy-1.1', 'ivy-1.2',
                'jedit-3.2', 'jedit-4.0', 'jedit-4.1', 'jedit-4.2', 'jedit-4.3',
                'log4j-1.0', 'log4j-1.1', 'log4j-1.2',
                'lucene-2.0', 'lucene-2.2', 'lucene-2.4',
                'poi-1.5', 'poi-2.0', 'poi-2.5', 'poi-3.0',
                'synapse-1.0', 'synapse-1.1', 'synapse-1.2',
                'velocity-1.4', 'velocity-1.5', 'velocity-1.6',
                'xalan-2.4', 'xalan-2.5', 'xalan-2.6', 'xalan-2.7',
                'xerces-1.1', 'xerces-1.2', 'xerces-1.3', 'xerces-1.4.4'],
    'LOC Moyen': [302, 304, 297, 323, 280, 99, 109, 112, 117, 246, 246, 249,
                  474, 473, 491, 465, 411, 160, 183, 186, 259, 257, 303,
                  234, 297, 311, 293, 183, 191, 209, 264, 248, 249,
                  311, 380, 465, 471, 560, 362, 369, 240],
    'WMC Moyen': [10.6, 10.5, 10.1, 11.1, 11.1, 8.1, 8.3, 8.5, 8.6,
                  10.6, 10.7, 11.3, 12.5, 12.9, 13.1, 13.2, 12.4,
                  6.6, 7.8, 8.4, 9.3, 9.4, 10.4, 13.4, 14.3, 14.3, 13.5,
                  8.1, 7.3, 7.6, 8.8, 9.0, 9.0, 11.4, 11.3, 11.0, 10.9,
                  11.4, 11.3, 11.4, 9.9],
    'NOC': [125, 178, 293, 351, 745, 339, 608, 872, 965, 111, 241, 352,
            272, 306, 312, 367, 492, 135, 109, 205, 195, 247, 340,
            237, 314, 385, 442, 157, 222, 256, 196, 214, 229,
            723, 803, 885, 909, 162, 440, 453, 588],
    'CBO Moyen': [10.4, 10.8, 10.6, 11.5, 11.0, 10.0, 10.1, 10.8, 11.1,
                  10.4, 11.3, 13.2, 12.0, 12.4, 13.0, 14.1, 14.3,
                  6.9, 6.9, 7.6, 9.8, 10.0, 10.8, 8.3, 8.7, 9.0, 10.1,
                  13.2, 12.5, 12.8, 10.6, 10.6, 10.8, 14.5, 12.9, 12.1, 12.0,
                  6.1, 4.9, 5.1, 6.3]
}

data = pd.DataFrame(data_dict)
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

pattern_colors = {
    'Stables': '#2E86AB',
    'Croissance': '#06A77D',
    'Oscillatoires': '#F18F01',
    'Décroissance': '#C73E1D'
}

print(f"✅ Données chargées: {data['Projet'].nunique()} projets")

def save_fig(fig, filename):
    """Sauvegarde PNG + PDF"""
    for fmt in ['png', 'pdf']:
        filepath = f"{filename}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=DPI, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"   ✅ {filepath}")

# ============================================================================
# FIGURE A.3.1 : COMPARAISON LOC vs WMC
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.1 : COMPARAISON LOC vs WMC")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=GRID_2x1)

# LOC
ax1 = axes[0]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax1.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax1.set_xlabel('Version', fontsize=9, fontweight='bold')
ax1.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
ax1.set_title('(a) Taille du code (LOC)', fontsize=10, fontweight='bold', pad=8)
ax1.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# WMC
ax2 = axes[1]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax2.plot(projet_data['Version_Num'], projet_data['WMC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax2.set_xlabel('Version', fontsize=9, fontweight='bold')
ax2.set_ylabel('WMC moyen', fontsize=9, fontweight='bold')
ax2.set_title('(b) Complexité (WMC)', fontsize=10, fontweight='bold', pad=8)
ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Comparaison LOC vs WMC par pattern',
             fontsize=11, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
save_fig(fig, 'Figure_A3_1_Comparaison_LOC_WMC')
plt.show()

# ============================================================================
# FIGURE A.3.2 : HEATMAP DES VARIATIONS
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.2 : HEATMAP DES VARIATIONS RELATIVES")
print("="*70)

variations = []
for projet in data['Projet'].unique():
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

    if len(projet_data) > 1:
        var_loc = ((projet_data.iloc[-1]['LOC Moyen'] - projet_data.iloc[0]['LOC Moyen'])
                   / projet_data.iloc[0]['LOC Moyen'] * 100)
        var_wmc = ((projet_data.iloc[-1]['WMC Moyen'] - projet_data.iloc[0]['WMC Moyen'])
                   / projet_data.iloc[0]['WMC Moyen'] * 100)
        var_noc = ((projet_data.iloc[-1]['NOC'] - projet_data.iloc[0]['NOC'])
                   / projet_data.iloc[0]['NOC'] * 100)
        var_cbo = ((projet_data.iloc[-1]['CBO Moyen'] - projet_data.iloc[0]['CBO Moyen'])
                   / projet_data.iloc[0]['CBO Moyen'] * 100)

        variations.append({
            'Projet': projet,
            'LOC': var_loc,
            'WMC': var_wmc,
            'NOC': var_noc,
            'CBO': var_cbo
        })

df_variations = pd.DataFrame(variations)
df_variations = df_variations.set_index('Projet')

fig, ax = plt.subplots(figsize=HEATMAP_FIG)

sns.heatmap(df_variations.T, annot=True, fmt='.0f', cmap='RdYlGn_r',
            center=0, cbar_kws={'label': 'Variation (%)', 'shrink': 0.8},
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 8})

ax.set_title('Variations relatives des métriques\n(première vs dernière version)',
             fontweight='bold', pad=12, fontsize=11)
ax.set_xlabel('Projets', fontweight='bold', fontsize=10)
ax.set_ylabel('Métriques', fontweight='bold', fontsize=10)

plt.tight_layout()
save_fig(fig, 'Figure_A3_2_Heatmap_Variations')
plt.show()

df_variations.to_csv('Tableau_A3_Variations_Detaillees.csv')
print("   ✅ Tableau_A3_Variations_Detaillees.csv")

# ============================================================================
# FIGURE A.3.3 : ÉVOLUTION NORMALISÉE (4 MÉTRIQUES)
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.3 : ÉVOLUTION NORMALISÉE - 4 MÉTRIQUES")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=NORMALIZED_FIG)

metrics_to_plot = ['LOC Moyen', 'WMC Moyen', 'NOC', 'CBO Moyen']
metric_labels = ['LOC moyen', 'WMC moyen', 'NOC', 'CBO moyen']

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[idx // 2, idx % 2]

    for pattern_name, projects in patterns.items():
        for projet in projects:
            projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

            # Normalisation min-max
            values = projet_data[metric].values
            if values.max() > values.min():
                normalized = (values - values.min()) / (values.max() - values.min())
            else:
                normalized = values * 0

            ax.plot(projet_data['Version_Num'], normalized,
                    marker='o', linewidth=1.5, markersize=4,
                    alpha=0.6, color=pattern_colors[pattern_name])

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('Valeur normalisée (0-1)', fontsize=9, fontweight='bold')
    ax.set_title(f'({chr(97+idx)}) {label}', fontsize=10, fontweight='bold', pad=8)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-0.05, 1.05)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Évolution normalisée des métriques par pattern\n' +
             '(valeurs normalisées 0-1 pour comparaison)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
save_fig(fig, 'Figure_A3_3_Evolution_Normalisee')
plt.show()

# ============================================================================
# RÉSUMÉ
# ============================================================================

print("\n" + "="*70)
print("✅ GÉNÉRATION TERMINÉE - ANNEXE A.3 (VERSION OPTIMALE)")
print("="*70)

print("\n📊 FICHIERS GÉNÉRÉS (PNG + PDF, 300 DPI) :")
print("   1. Figure_A3_1_Comparaison_LOC_WMC (6.5\" × 6\")")
print("   2. Figure_A3_2_Heatmap_Variations (6.5\" × 4.5\")")
print("   3. Figure_A3_3_Evolution_Normalisee (6.5\" × 8\")")
print("   4. Tableau_A3_Variations_Detaillees.csv")

print("\n💡 STRUCTURE OPTIMALE :")
print("   • Chapitre 4 : Figure 4.X (vue d'ensemble 2×2)")
print("   • Annexe A.3 : 3 figures complémentaires")
print("   • Gain d'espace : 4 figures superflues éliminées")

print("\n📝 LÉGENDES POUR ANNEXE A.3 :")

legendes = """
Figure A.3.1 – Comparaison de l'évolution de la taille du code (LOC moyen)
et de la complexité (WMC moyen) pour les quatre patterns. Les projets en
croissance montrent une évolution coordonnée des deux métriques, tandis que
les projets en décroissance présentent une dissociation suggérant des efforts
de refactoring ciblés.

Figure A.3.2 – Heatmap des variations relatives (%) des métriques entre
première et dernière version pour LOC, WMC, NOC et CBO. Les valeurs positives
(teintes rouges) indiquent une augmentation, les valeurs négatives (teintes
vertes) une diminution. Les projets en croissance (Lucene, Xalan, Synapse)
montrent des augmentations coordonnées de LOC et NOC, tandis que les projets
en décroissance (JEdit, Xerces) présentent des patterns hétérogènes suggérant
des efforts de refactoring ciblés.

Figure A.3.3 – Évolution normalisée des quatre métriques principales
(LOC, WMC, NOC, CBO) pour l'ensemble des projets. Les valeurs sont normalisées
entre 0 et 1 pour permettre la comparaison directe. (a) LOC moyen : les
patterns de croissance montrent une évolution monotone croissante, les patterns
stables restent dans une bande étroite, les patterns oscillatoires exhibent
des fluctuations périodiques, et les patterns de décroissance montrent une
tendance décroissante globale. (b) WMC moyen : évolution plus stable que LOC,
reflétant la dissociation entre taille et complexité. (c) NOC : corrélation
forte avec LOC (ρ > 0.8), reflétant la croissance structurelle des projets.
(d) CBO moyen : patterns plus hétérogènes indiquant des stratégies
architecturales différentes selon les projets.
"""

print(legendes)

print("\n" + "="*70)

In [ ]:
"""
================================================================================
FIGURE 4.X - PATTERNS D'ÉVOLUTION TEMPORELLE (CHAPITRE 4)
================================================================================

Génère la figure principale à insérer dans le Chapitre 4 (section 4.1.2)
Vue d'ensemble 2×2 des 4 patterns d'évolution

Taille optimisée pour Word (6.5" × 8")
Résolution 300 DPI
Format PNG + PDF

Auteure: Mireille Kafando
Institution: UQTR
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

# Taille pour figure pleine page Word
FIG_SIZE = (6.5, 8)
DPI = 300

plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

print("✅ Configuration chargée (300 DPI)")

# ============================================================================
# DONNÉES
# ============================================================================

data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'LOC Moyen': [302, 304, 297, 323, 280, 99, 109, 112, 117, 246, 246, 249,
                  474, 473, 491, 465, 411, 160, 183, 186, 259, 257, 303,
                  234, 297, 311, 293, 183, 191, 209, 264, 248, 249,
                  311, 380, 465, 471, 560, 362, 369, 240],
}

data = pd.DataFrame(data_dict)
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

pattern_colors = {
    'Stables': '#2E86AB',
    'Croissance': '#06A77D',
    'Oscillatoires': '#F18F01',
    'Décroissance': '#C73E1D'
}

print(f"✅ Données chargées: {data['Projet'].nunique()} projets")

# ============================================================================
# FIGURE 4.X : VUE D'ENSEMBLE 2×2
# ============================================================================

print("\n" + "="*70)
print("GÉNÉRATION : FIGURE 4.X - PATTERNS D'ÉVOLUTION TEMPORELLE")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=FIG_SIZE)
axes = axes.ravel()

for idx, (pattern_name, projects) in enumerate(patterns.items()):
    ax = axes[idx]

    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=2, markersize=5, label=projet, alpha=0.85)

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
    ax.set_title(f'{pattern_name}', fontsize=10, fontweight='bold',
                 pad=8, color=pattern_colors[pattern_name])
    ax.legend(loc='best', frameon=True, shadow=False, fontsize=7)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Patterns d\'évolution temporelle des métriques (LOC moyen)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()

# Sauvegarder
for fmt in ['png', 'pdf']:
    filename = f'Figure_4_X_Patterns_Evolution_Temporelle.{fmt}'
    fig.savefig(filename, format=fmt, dpi=DPI, bbox_inches='tight',
               facecolor='white', edgecolor='none')
    print(f"   ✅ {filename}")

plt.show()

# ============================================================================
# LÉGENDE POUR LE MÉMOIRE
# ============================================================================

print("\n" + "="*70)
print("LÉGENDE À COPIER DANS LE CHAPITRE 4")
print("="*70)

legende = """
Figure 4.X – Patterns d'évolution temporelle des métriques LOC dans le
corpus PROMISE. Les quatre patterns identifiés présentent des caractéristiques
distinctes qui justifient notre évaluation comparative de trois stratégies de
standardisation (SG, SV, SR-Robuste) : (a) Stables – métriques relativement
constantes entre versions (Ant, Log4j, POI, Ivy) suggérant l'efficacité de
la standardisation globale SG ; (b) Croissance – tendances croissantes
systématiques (Lucene, Xalan, Synapse) motivant l'adaptation par version SV
qui s'adapte aux caractéristiques locales ; (c) Oscillatoires – variations
périodiques (Velocity, Camel) rendant problématiques les approches purement
globales ou locales et motivant la stratégie hybride SR-Robuste ;
(d) Décroissance – amélioration continue (JEdit, Xerces) suggérant des
efforts de refactoring et renforçant l'importance du respect de la chronologie
temporelle. Les analyses détaillées incluant WMC, CBO et les évolutions
normalisées sont présentées en Annexe A.3.
"""

print(legende)

print("\n" + "="*70)
print("INSERTION DANS LE TEXTE")
print("="*70)

insertion = """
EMPLACEMENT : Chapitre 4, Section 4.1.2, après le paragraphe
"Patterns d'évolution temporelle", avant "Corrélations inter-métriques"

TEXTE MODIFIÉ :
---------------
...renforçant l'importance de respecter scrupuleusement la chronologie
temporelle dans nos protocoles de validation pour simuler des conditions
de déploiement réalistes (section 4.4). La Figure 4.X synthétise
visuellement ces quatre patterns d'évolution, illustrant les différences
comportementales qui sous-tendent notre approche différenciée de
standardisation.

[INSÉRER FIGURE 4.X ICI]

Figure 4.X – Patterns d'évolution temporelle...

Corrélations inter-métriques : Les métriques CK que nous utilisons...
"""

print(insertion)

print("\n" + "="*70)
print("✅ GÉNÉRATION TERMINÉE")
print("="*70)
print("\n📊 FICHIER GÉNÉRÉ :")
print("   • Figure_4_X_Patterns_Evolution_Temporelle.png (6.5\" × 8\")")
print("   • Figure_4_X_Patterns_Evolution_Temporelle.pdf (backup)")
print("\n🎯 À insérer dans : Chapitre 4, section 4.1.2")
print("📄 Résolution : 300 DPI (qualité impression)")
print("="*70)

In [ ]:
"""
================================================================================
ANNEXE A.3 - ANALYSES COMPLÉMENTAIRES (VERSION OPTIMALE)
================================================================================

Génère les 3 figures essentielles pour l'Annexe A.3 :
1. Comparaison LOC vs WMC (validation sur 2 métriques)
2. Heatmap des variations (quantification précise)
3. Évolution normalisée 4 métriques (validation robustesse)

Note : La vue d'ensemble 2×2 est maintenant Figure 4.X dans le texte principal
Les détails par pattern (anciennes A.3.2-5) sont superflus car synthétisés
dans Figure 4.X

Tailles optimisées pour Word
Résolution 300 DPI

Auteure: Mireille Kafando
Institution: UQTR
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

# Tailles optimisées
GRID_2x1 = (6.5, 6)
HEATMAP_FIG = (6.5, 4.5)
NORMALIZED_FIG = (6.5, 8)
DPI = 300

plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'figure.dpi': DPI,
    'savefig.dpi': DPI,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

print("✅ Configuration chargée")

# ============================================================================
# DONNÉES
# ============================================================================

data_dict = {
    'Projet': ['Ant', 'Ant', 'Ant', 'Ant', 'Ant',
               'Camel', 'Camel', 'Camel', 'Camel',
               'Ivy', 'Ivy', 'Ivy',
               'Jedit', 'Jedit', 'Jedit', 'Jedit', 'Jedit',
               'Log4j', 'Log4j', 'Log4j',
               'Lucene', 'Lucene', 'Lucene',
               'Poi', 'Poi', 'Poi', 'Poi',
               'Synapse', 'Synapse', 'Synapse',
               'Velocity', 'Velocity', 'Velocity',
               'Xalan', 'Xalan', 'Xalan', 'Xalan',
               'Xerces', 'Xerces', 'Xerces', 'Xerces'],
    'Version': ['ant-1.3', 'ant-1.4', 'ant-1.5', 'ant-1.6', 'ant-1.7',
                'camel-1.0', 'camel-1.2', 'camel-1.4', 'camel-1.6',
                'ivy-1.0', 'ivy-1.1', 'ivy-1.2',
                'jedit-3.2', 'jedit-4.0', 'jedit-4.1', 'jedit-4.2', 'jedit-4.3',
                'log4j-1.0', 'log4j-1.1', 'log4j-1.2',
                'lucene-2.0', 'lucene-2.2', 'lucene-2.4',
                'poi-1.5', 'poi-2.0', 'poi-2.5', 'poi-3.0',
                'synapse-1.0', 'synapse-1.1', 'synapse-1.2',
                'velocity-1.4', 'velocity-1.5', 'velocity-1.6',
                'xalan-2.4', 'xalan-2.5', 'xalan-2.6', 'xalan-2.7',
                'xerces-1.1', 'xerces-1.2', 'xerces-1.3', 'xerces-1.4.4'],
    'LOC Moyen': [302, 304, 297, 323, 280, 99, 109, 112, 117, 246, 246, 249,
                  474, 473, 491, 465, 411, 160, 183, 186, 259, 257, 303,
                  234, 297, 311, 293, 183, 191, 209, 264, 248, 249,
                  311, 380, 465, 471, 560, 362, 369, 240],
    'WMC Moyen': [10.6, 10.5, 10.1, 11.1, 11.1, 8.1, 8.3, 8.5, 8.6,
                  10.6, 10.7, 11.3, 12.5, 12.9, 13.1, 13.2, 12.4,
                  6.6, 7.8, 8.4, 9.3, 9.4, 10.4, 13.4, 14.3, 14.3, 13.5,
                  8.1, 7.3, 7.6, 8.8, 9.0, 9.0, 11.4, 11.3, 11.0, 10.9,
                  11.4, 11.3, 11.4, 9.9],
    'NOC': [125, 178, 293, 351, 745, 339, 608, 872, 965, 111, 241, 352,
            272, 306, 312, 367, 492, 135, 109, 205, 195, 247, 340,
            237, 314, 385, 442, 157, 222, 256, 196, 214, 229,
            723, 803, 885, 909, 162, 440, 453, 588],
    'CBO Moyen': [10.4, 10.8, 10.6, 11.5, 11.0, 10.0, 10.1, 10.8, 11.1,
                  10.4, 11.3, 13.2, 12.0, 12.4, 13.0, 14.1, 14.3,
                  6.9, 6.9, 7.6, 9.8, 10.0, 10.8, 8.3, 8.7, 9.0, 10.1,
                  13.2, 12.5, 12.8, 10.6, 10.6, 10.8, 14.5, 12.9, 12.1, 12.0,
                  6.1, 4.9, 5.1, 6.3]
}

data = pd.DataFrame(data_dict)
data['Version_Num'] = data.groupby('Projet').cumcount() + 1

patterns = {
    'Stables': ['Ant', 'Log4j', 'Poi', 'Ivy'],
    'Croissance': ['Lucene', 'Xalan', 'Synapse'],
    'Oscillatoires': ['Velocity', 'Camel'],
    'Décroissance': ['Jedit', 'Xerces']
}

pattern_colors = {
    'Stables': '#2E86AB',
    'Croissance': '#06A77D',
    'Oscillatoires': '#F18F01',
    'Décroissance': '#C73E1D'
}

print(f"✅ Données chargées: {data['Projet'].nunique()} projets")

def save_fig(fig, filename):
    """Sauvegarde PNG + PDF"""
    for fmt in ['png', 'pdf']:
        filepath = f"{filename}.{fmt}"
        fig.savefig(filepath, format=fmt, dpi=DPI, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"   ✅ {filepath}")

# ============================================================================
# FIGURE A.3.1 : COMPARAISON LOC vs WMC
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.1 : COMPARAISON LOC vs WMC")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=GRID_2x1)

# LOC
ax1 = axes[0]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax1.plot(projet_data['Version_Num'], projet_data['LOC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax1.set_xlabel('Version', fontsize=9, fontweight='bold')
ax1.set_ylabel('LOC moyen', fontsize=9, fontweight='bold')
ax1.set_title('(a) Taille du code (LOC)', fontsize=10, fontweight='bold', pad=8)
ax1.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# WMC
ax2 = axes[1]
for pattern_name, projects in patterns.items():
    for projet in projects:
        projet_data = data[data['Projet'] == projet].sort_values('Version_Num')
        ax2.plot(projet_data['Version_Num'], projet_data['WMC Moyen'],
                marker='o', linewidth=1.5, markersize=4,
                alpha=0.6, color=pattern_colors[pattern_name])

ax2.set_xlabel('Version', fontsize=9, fontweight='bold')
ax2.set_ylabel('WMC moyen', fontsize=9, fontweight='bold')
ax2.set_title('(b) Complexité (WMC)', fontsize=10, fontweight='bold', pad=8)
ax2.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Comparaison LOC vs WMC par pattern',
             fontsize=11, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
save_fig(fig, 'Figure_A3_1_Comparaison_LOC_WMC')
plt.show()

# ============================================================================
# FIGURE A.3.2 : HEATMAP DES VARIATIONS
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.2 : HEATMAP DES VARIATIONS RELATIVES")
print("="*70)

variations = []
for projet in data['Projet'].unique():
    projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

    if len(projet_data) > 1:
        var_loc = ((projet_data.iloc[-1]['LOC Moyen'] - projet_data.iloc[0]['LOC Moyen'])
                   / projet_data.iloc[0]['LOC Moyen'] * 100)
        var_wmc = ((projet_data.iloc[-1]['WMC Moyen'] - projet_data.iloc[0]['WMC Moyen'])
                   / projet_data.iloc[0]['WMC Moyen'] * 100)
        var_noc = ((projet_data.iloc[-1]['NOC'] - projet_data.iloc[0]['NOC'])
                   / projet_data.iloc[0]['NOC'] * 100)
        var_cbo = ((projet_data.iloc[-1]['CBO Moyen'] - projet_data.iloc[0]['CBO Moyen'])
                   / projet_data.iloc[0]['CBO Moyen'] * 100)

        variations.append({
            'Projet': projet,
            'LOC': var_loc,
            'WMC': var_wmc,
            'NOC': var_noc,
            'CBO': var_cbo
        })

df_variations = pd.DataFrame(variations)
df_variations = df_variations.set_index('Projet')

fig, ax = plt.subplots(figsize=HEATMAP_FIG)

sns.heatmap(df_variations.T, annot=True, fmt='.0f', cmap='RdYlGn_r',
            center=0, cbar_kws={'label': 'Variation (%)', 'shrink': 0.8},
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 8})

ax.set_title('Variations relatives des métriques\n(première vs dernière version)',
             fontweight='bold', pad=12, fontsize=11)
ax.set_xlabel('Projets', fontweight='bold', fontsize=10)
ax.set_ylabel('Métriques', fontweight='bold', fontsize=10)

plt.tight_layout()
save_fig(fig, 'Figure_A3_2_Heatmap_Variations')
plt.show()

df_variations.to_csv('Tableau_A3_Variations_Detaillees.csv')
print("   ✅ Tableau_A3_Variations_Detaillees.csv")

# ============================================================================
# FIGURE A.3.3 : ÉVOLUTION NORMALISÉE (4 MÉTRIQUES)
# ============================================================================

print("\n" + "="*70)
print("FIGURE A.3.3 : ÉVOLUTION NORMALISÉE - 4 MÉTRIQUES")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=NORMALIZED_FIG)

metrics_to_plot = ['LOC Moyen', 'WMC Moyen', 'NOC', 'CBO Moyen']
metric_labels = ['LOC moyen', 'WMC moyen', 'NOC', 'CBO moyen']

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[idx // 2, idx % 2]

    for pattern_name, projects in patterns.items():
        for projet in projects:
            projet_data = data[data['Projet'] == projet].sort_values('Version_Num')

            # Normalisation min-max
            values = projet_data[metric].values
            if values.max() > values.min():
                normalized = (values - values.min()) / (values.max() - values.min())
            else:
                normalized = values * 0

            ax.plot(projet_data['Version_Num'], normalized,
                    marker='o', linewidth=1.5, markersize=4,
                    alpha=0.6, color=pattern_colors[pattern_name])

    ax.set_xlabel('Version', fontsize=9, fontweight='bold')
    ax.set_ylabel('Valeur normalisée (0-1)', fontsize=9, fontweight='bold')
    ax.set_title(f'({chr(97+idx)}) {label}', fontsize=10, fontweight='bold', pad=8)
    ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-0.05, 1.05)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=pattern, alpha=0.8)
                  for pattern, color in pattern_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
          frameon=True, fontsize=8, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Évolution normalisée des métriques par pattern\n' +
             '(valeurs normalisées 0-1 pour comparaison)',
             fontsize=11, fontweight='bold', y=0.995)
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
save_fig(fig, 'Figure_A3_3_Evolution_Normalisee')
plt.show()

# ============================================================================
# RÉSUMÉ
# ============================================================================

print("\n" + "="*70)
print("✅ GÉNÉRATION TERMINÉE - ANNEXE A.3 (VERSION OPTIMALE)")
print("="*70)

print("\n📊 FICHIERS GÉNÉRÉS (PNG + PDF, 300 DPI) :")
print("   1. Figure_A3_1_Comparaison_LOC_WMC (6.5\" × 6\")")
print("   2. Figure_A3_2_Heatmap_Variations (6.5\" × 4.5\")")
print("   3. Figure_A3_3_Evolution_Normalisee (6.5\" × 8\")")
print("   4. Tableau_A3_Variations_Detaillees.csv")

print("\n💡 STRUCTURE OPTIMALE :")
print("   • Chapitre 4 : Figure 4.X (vue d'ensemble 2×2)")
print("   • Annexe A.3 : 3 figures complémentaires")
print("   • Gain d'espace : 4 figures superflues éliminées")

print("\n📝 LÉGENDES POUR ANNEXE A.3 :")

legendes = """
Figure A.3.1 – Comparaison de l'évolution de la taille du code (LOC moyen)
et de la complexité (WMC moyen) pour les quatre patterns. Les projets en
croissance montrent une évolution coordonnée des deux métriques, tandis que
les projets en décroissance présentent une dissociation suggérant des efforts
de refactoring ciblés.

Figure A.3.2 – Heatmap des variations relatives (%) des métriques entre
première et dernière version pour LOC, WMC, NOC et CBO. Les valeurs positives
(teintes rouges) indiquent une augmentation, les valeurs négatives (teintes
vertes) une diminution. Les projets en croissance (Lucene, Xalan, Synapse)
montrent des augmentations coordonnées de LOC et NOC, tandis que les projets
en décroissance (JEdit, Xerces) présentent des patterns hétérogènes suggérant
des efforts de refactoring ciblés.

Figure A.3.3 – Évolution normalisée des quatre métriques principales
(LOC, WMC, NOC, CBO) pour l'ensemble des projets. Les valeurs sont normalisées
entre 0 et 1 pour permettre la comparaison directe. (a) LOC moyen : les
patterns de croissance montrent une évolution monotone croissante, les patterns
stables restent dans une bande étroite, les patterns oscillatoires exhibent
des fluctuations périodiques, et les patterns de décroissance montrent une
tendance décroissante globale. (b) WMC moyen : évolution plus stable que LOC,
reflétant la dissociation entre taille et complexité. (c) NOC : corrélation
forte avec LOC (ρ > 0.8), reflétant la croissance structurelle des projets.
(d) CBO moyen : patterns plus hétérogènes indiquant des stratégies
architecturales différentes selon les projets.
"""

print(legendes)

print("\n" + "="*70)